# songforge — vocal in, song out

Upload a vocal stem, get a finished song back: the vocal is analysed (tempo,
key, melody, structure), a backing track is composed against those
constraints, then both are mixed and mastered.

Runtime → **Change runtime type** → CPU is enough for this notebook (≈1 min
for a 30 s song). Switch to a GPU only if you later plug in MusicGen or a
trained checkpoint.

All files are temporary: they live in `/content` and disappear when the
runtime ends.

In [ ]:
# install songforge straight from the repository (numpy + soundfile only)
!pip install -q "songforge[io] @ git+https://github.com/SKYDO234/SKVM.git#subdirectory=songforge"
import songforge, numpy, sys
print("songforge", songforge.__version__, "| python", sys.version.split()[0])

## 1. Proof it works: a synthetic vocal → full song

No assets needed — this builds a small melody, then runs the whole pipeline.

In [ ]:
from songforge.cli import main
main(["demo", "-o", "/content/demo_out", "--bpm", "104"])

In [ ]:
from IPython.display import Audio
for name in ("song", "instrumental", "vocals"):
    print(name)
    display(Audio(f"/content/demo_out/{name}.wav"))

## 2. Your own vocal

Upload a **clean vocal stem** (WAV/MP3/FLAC). If you only have a full song,
install Demucs first (`!pip install -q demucs`) and pass `--input-kind song`
instead — separation adds a few minutes on CPU.

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick your vocal file
vocal_path = list(uploaded)[0]
print("uploaded:", vocal_path)

In [ ]:
from songforge.pipeline import PipelineConfig, run_pipeline

cfg = PipelineConfig(
    input_path=vocal_path,
    out_dir="/content/out",
    input_kind="vocals",           # use "song" if you uploaded a full mix (needs demucs)
    backend="procedural",          # or "musicgen" on a GPU runtime
    style="auto",                  # lofi | house | trap | rock | ballad | ambient
    prompt="warm, dreamy, lo-fi",   # free-text style hint
    seed=7,
    max_seconds=60,                # keep the free runtime happy
)
result = run_pipeline(cfg)
print(result.master_path)

In [ ]:
from IPython.display import Audio
from songforge.pipeline import summarize_report

print(summarize_report("/content/out/report.json"))
display(Audio("/content/out/song.wav"))

In [ ]:
# download everything (song + instrumental + vocals + report) as one zip
import shutil
from google.colab import files

shutil.make_archive("/content/songforge_result", "zip", "/content/out")
files.download("/content/songforge_result.zip")

## 3. If you want more than the CPU backend

* **GPU + a pretrained model**: `!pip install -q torch audiocraft`, then
  `backend="musicgen"` above (30 s windows, needs GPU memory).
* **Your own trained model**: see `songforge/train/` — `prepare_dataset` →
  `train` → `infer`, then either upload the checkpoint and use
  `backend="external"` with a `command=` template, or call
  `songforge_train.infer` directly.
* **A public web app for others**: `songforge serve` (see `DEPLOY.md`).